# 01 - PySpark Basics

This notebook teaches you PySpark fundamentals through hands-on exercises. By the end, you'll understand:
- How to create a SparkSession
- How DataFrames work (and how they differ from Pandas)
- Transformations vs Actions (lazy evaluation)
- The core operations you'll use in the medallion pipeline
- User Defined Functions (UDFs)

**How to use this notebook:** Run each cell, read the output, then complete the exercises marked with `# YOUR TURN`. Don't skip ahead -- each section builds on the previous one.

---
## 1. Creating a SparkSession

Every PySpark program starts here. The `SparkSession` is your connection to the Spark engine. Think of it like opening a database connection -- nothing happens until you have one.

- `.appName()` is just a label (shows up in the Spark UI)
- `.master("local[*]")` means "run on this machine, use all CPU cores"
- `.getOrCreate()` reuses an existing session if one is already running

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark-Basics") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"App name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/06 07:24:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.3
App name: PySpark-Basics
Master: local[*]


26/08/06 07:24:26 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


---
## 2. Creating DataFrames

A DataFrame is a table with named columns and typed rows. You can create one from:
- Python lists/dicts (for testing)
- Files on disk (JSON, CSV, Parquet)

Let's start with Python data so you can see exactly what's happening.

In [2]:
# Creating a DataFrame from a list of tuples
data = [
    ("Gasabo", 6.5, 2300),
    ("Kicukiro", 5.8, 1800),
    ("Musanze", 7.1, 3200),
    ("Huye", 6.9, 2900),
    ("Nyarugenge", 5.2, 1500),
]

columns = ["district", "soil_ph", "yield_kg"]

df = spark.createDataFrame(data, columns)
df.show()

+----------+-------+--------+
|  district|soil_ph|yield_kg|
+----------+-------+--------+
|    Gasabo|    6.5|    2300|
|  Kicukiro|    5.8|    1800|
|   Musanze|    7.1|    3200|
|      Huye|    6.9|    2900|
|Nyarugenge|    5.2|    1500|
+----------+-------+--------+



In [3]:
# Inspect the schema -- Spark inferred the column types from your Python data
df.printSchema()

root
 |-- district: string (nullable = true)
 |-- soil_ph: double (nullable = true)
 |-- yield_kg: long (nullable = true)



In [4]:
# You can also define the schema explicitly
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("district", StringType(), nullable=False),
    StructField("soil_ph", DoubleType(), nullable=True),
    StructField("yield_kg", IntegerType(), nullable=True),
])

df_typed = spark.createDataFrame(data, schema)
df_typed.printSchema()

root
 |-- district: string (nullable = false)
 |-- soil_ph: double (nullable = true)
 |-- yield_kg: integer (nullable = true)



**Why define schemas explicitly?**

With small data, Spark can infer types. With large files, inference means Spark reads the data twice (once to guess types, once to load). Explicit schemas skip that first pass. In your pipeline, you'll use explicit schemas for reliability -- if incoming data has wrong types, you want it to fail immediately, not silently convert.

### YOUR TURN

Create a DataFrame with 4 rows representing farmers. Include columns: `name` (string), `phone` (string), `district` (string), `hectares` (double). Use explicit schema.

In [9]:
# YOUR TURN: Create the farmer DataFrame here
farmer_data = [
    ("Mugabo", "+250787909098", "Gasabo", 500.0),
    ("Jean-Bosco", "+250787909098", "Nyarugenge", 10.0),
    ("Matayo", "+250787909098", "Musanze", 200.0),
    ("Shyaka", "+250787909098", "Rubavu", 10.5),
    ("Rugero", "+250787909098", "Nyabihu", 40.2),
    
]
farmer_schema = StructType([
    StructField("name", StringType(), nullable=False),
    StructField("phone", StringType(), nullable=True),
    StructField("district", StringType(), nullable=False),
    StructField("hectares", DoubleType(), nullable=False)
])

df_farmers = spark.createDataFrame(farmer_data, farmer_schema)
df_farmers.printSchema()



root
 |-- name: string (nullable = false)
 |-- phone: string (nullable = true)
 |-- district: string (nullable = false)
 |-- hectares: double (nullable = false)



---
## 3. Reading Data from Files

In the real pipeline, you'll read JSON files produced by the mock data generator. Let's create a small JSON file and read it.

In [15]:
import json
import os

# Create a small sample JSON file
sample_data = [
    {"record_id": "r001", "national_id": "1199580012345", "full_name": "Alice Mukamana",
     "phone_number": "+250781234567", "district": "Gasabo", "soil_ph": 6.5, "yield_estimate_kg": 2300},
    {"record_id": "r002", "national_id": "1199580067890", "full_name": "Jean Habimana",
     "phone_number": "+250789876543", "district": "Kicukiro", "soil_ph": 5.8, "yield_estimate_kg": 1800},
    {"record_id": "r003", "national_id": "1199580011111", "full_name": "Marie Uwimana",
     "phone_number": "+250782222222", "district": "Gasabo", "soil_ph": 7.1, "yield_estimate_kg": 3200},
    {"record_id": "r004", "national_id": "1199580033333", "full_name": "Eric Niyonzima",
     "phone_number": "+250784444444", "district": "Musanze", "soil_ph": 6.2, "yield_estimate_kg": 2700},
]

os.makedirs("/home/jovyan/data/raw", exist_ok=True)
with open("/home/jovyan/data/raw/sample.json", "w") as f:
    for record in sample_data:
        f.write(json.dumps(record) + "\n")

print("Wrote sample.json")

Wrote sample.json


In [16]:
# Read the JSON file into a DataFrame
json_df = spark.read.json("/home/jovyan/data/raw/sample.json")
json_df.show(truncate=False)
json_df.printSchema()

+--------+--------------+-------------+-------------+---------+-------+-----------------+
|district|full_name     |national_id  |phone_number |record_id|soil_ph|yield_estimate_kg|
+--------+--------------+-------------+-------------+---------+-------+-----------------+
|Gasabo  |Alice Mukamana|1199580012345|+250781234567|r001     |6.5    |2300             |
|Kicukiro|Jean Habimana |1199580067890|+250789876543|r002     |5.8    |1800             |
|Gasabo  |Marie Uwimana |1199580011111|+250782222222|r003     |7.1    |3200             |
|Musanze |Eric Niyonzima|1199580033333|+250784444444|r004     |6.2    |2700             |
+--------+--------------+-------------+-------------+---------+-------+-----------------+

root
 |-- district: string (nullable = true)
 |-- full_name: string (nullable = true)
 |-- national_id: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- record_id: string (nullable = true)
 |-- soil_ph: double (nullable = true)
 |-- yield_estimate_kg: lo

Notice that Spark inferred all the types from the JSON. `national_id` became a string (because of the leading digits), `soil_ph` became a double, `yield_estimate_kg` became a long (Spark's default integer).

This `json_df` is what your Bronze layer will look like -- raw data, all columns present, including PII.

---
## 4. Transformations -- Building a Plan (Lazy)

This is the most important concept in PySpark.

When you call `.select()`, `.filter()`, `.withColumn()`, etc., **nothing executes**. Spark just records what you want to do. It builds a plan called a DAG (directed acyclic graph).

Let's prove it.

In [17]:
from pyspark.sql.functions import col, upper

# These three lines do ZERO work -- they just build a plan
step1 = json_df.select("district", "soil_ph", "yield_estimate_kg")
step2 = step1.filter(col("soil_ph") > 6.0)
step3 = step2.withColumn("district_upper", upper(col("district")))

# Proof: check the type -- it's still a DataFrame, not actual data
print(type(step3))
print("No computation happened yet.")

<class 'pyspark.sql.dataframe.DataFrame'>
No computation happened yet.


In [18]:
# Now let's see the plan Spark built
step3.explain()

== Physical Plan ==
*(1) Project [district#75, soil_ph#80, yield_estimate_kg#81L, upper(district#75) AS district_upper#129]
+- *(1) Filter (isnotnull(soil_ph#80) AND (soil_ph#80 > 6.0))
   +- FileScan json [district#75,soil_ph#80,yield_estimate_kg#81L] Batched: false, DataFilters: [isnotnull(soil_ph#80), (soil_ph#80 > 6.0)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/data/raw/sample.json], PartitionFilters: [], PushedFilters: [IsNotNull(soil_ph), GreaterThan(soil_ph,6.0)], ReadSchema: struct<district:string,soil_ph:double,yield_estimate_kg:bigint>




Read the plan bottom-to-top:
1. Scan the JSON file
2. Apply the filter (`soil_ph > 6.0`)
3. Project (select) the columns + compute `upper(district)`

Spark rearranged things for efficiency -- it filters before projecting, even though we wrote `select` first. This is the **Catalyst optimizer** at work.

In [19]:
# NOW trigger execution with an action
step3.show()

+--------+-------+-----------------+--------------+
|district|soil_ph|yield_estimate_kg|district_upper|
+--------+-------+-----------------+--------------+
|  Gasabo|    6.5|             2300|        GASABO|
|  Gasabo|    7.1|             3200|        GASABO|
| Musanze|    6.2|             2700|       MUSANZE|
+--------+-------+-----------------+--------------+



### Key takeaway

| Type | Examples | What happens |
|------|---------|-------------|
| **Transformation** (lazy) | `select`, `filter`, `withColumn`, `groupBy`, `join`, `drop`, `orderBy` | Builds a plan, returns a new DataFrame |
| **Action** (eager) | `show`, `count`, `collect`, `write`, `first`, `take` | Executes the plan, returns results |

If you chain 20 transformations, Spark does nothing until you call an action. Then it optimizes the whole chain and runs it.

### YOUR TURN

Starting from `json_df`:
1. Filter to only rows where `district` is `"Gasabo"`
2. Select only `record_id`, `soil_ph`, and `yield_estimate_kg`
3. Call `.explain()` to see the plan
4. Call `.show()` to see the results

Question to answer: did Spark rearrange your filter and select? Check the explain output.

In [22]:
# YOUR TURN: filter + select + explain + show
transf1 = json_df.filter(col("district") == "Gasabo")
transf2 = transf1.select("record_id", "soil_ph", "yield_estimate_kg")

transf2.explain()
transf2.show()


== Physical Plan ==
*(1) Project [record_id#79, soil_ph#80, yield_estimate_kg#81L]
+- *(1) Filter (isnotnull(district#75) AND (district#75 = Gasabo))
   +- FileScan json [district#75,record_id#79,soil_ph#80,yield_estimate_kg#81L] Batched: false, DataFilters: [isnotnull(district#75), (district#75 = Gasabo)], Format: JSON, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/data/raw/sample.json], PartitionFilters: [], PushedFilters: [IsNotNull(district), EqualTo(district,Gasabo)], ReadSchema: struct<district:string,record_id:string,soil_ph:double,yield_estimate_kg:bigint>


+---------+-------+-----------------+
|record_id|soil_ph|yield_estimate_kg|
+---------+-------+-----------------+
|     r001|    6.5|             2300|
|     r003|    7.1|             3200|
+---------+-------+-----------------+



---
## 5. Core DataFrame Operations

These are the operations you'll use in the pipeline. Let's go through each one.

### 5a. select -- pick columns

In [23]:
# Two ways to reference columns
json_df.select("district", "soil_ph").show()        # by name (string)
json_df.select(col("district"), col("soil_ph")).show()  # by col() -- needed for expressions

+--------+-------+
|district|soil_ph|
+--------+-------+
|  Gasabo|    6.5|
|Kicukiro|    5.8|
|  Gasabo|    7.1|
| Musanze|    6.2|
+--------+-------+

+--------+-------+
|district|soil_ph|
+--------+-------+
|  Gasabo|    6.5|
|Kicukiro|    5.8|
|  Gasabo|    7.1|
| Musanze|    6.2|
+--------+-------+



### 5b. filter / where -- keep matching rows

In [24]:
# filter and where are identical -- use whichever reads better
json_df.filter(col("yield_estimate_kg") > 2500).show()

# Multiple conditions: use & (and), | (or), ~ (not) -- wrap each in parentheses
json_df.filter(
    (col("soil_ph") > 6.0) & (col("district") != "Kicukiro")
).show()

+--------+--------------+-------------+-------------+---------+-------+-----------------+
|district|     full_name|  national_id| phone_number|record_id|soil_ph|yield_estimate_kg|
+--------+--------------+-------------+-------------+---------+-------+-----------------+
|  Gasabo| Marie Uwimana|1199580011111|+250782222222|     r003|    7.1|             3200|
| Musanze|Eric Niyonzima|1199580033333|+250784444444|     r004|    6.2|             2700|
+--------+--------------+-------------+-------------+---------+-------+-----------------+

+--------+--------------+-------------+-------------+---------+-------+-----------------+
|district|     full_name|  national_id| phone_number|record_id|soil_ph|yield_estimate_kg|
+--------+--------------+-------------+-------------+---------+-------+-----------------+
|  Gasabo|Alice Mukamana|1199580012345|+250781234567|     r001|    6.5|             2300|
|  Gasabo| Marie Uwimana|1199580011111|+250782222222|     r003|    7.1|             3200|
| Musanze

### 5c. withColumn -- add or transform a column

This is critical for the pipeline: you'll use it to add hashed IDs, timestamps, and compliance flags.

In [25]:
from pyspark.sql.functions import lit, current_timestamp

# Add a constant column
df_with_flag = json_df.withColumn("compliance_flag", lit("RW_LAW_058_2021"))

# Add a computed column
df_with_yield_tons = json_df.withColumn("yield_tons", col("yield_estimate_kg") / 1000)

# Add a timestamp
df_with_ts = json_df.withColumn("processed_at", current_timestamp())

df_with_ts.select("record_id", "district", "processed_at").show(truncate=False)

+---------+--------+--------------------------+
|record_id|district|processed_at              |
+---------+--------+--------------------------+
|r001     |Gasabo  |2026-08-06 08:03:54.779259|
|r002     |Kicukiro|2026-08-06 08:03:54.779259|
|r003     |Gasabo  |2026-08-06 08:03:54.779259|
|r004     |Musanze |2026-08-06 08:03:54.779259|
+---------+--------+--------------------------+



**Important:** `withColumn` does NOT modify the original DataFrame. DataFrames are immutable. It returns a new one.

```python
# This does nothing useful:
json_df.withColumn("new_col", lit(1))  # result is thrown away

# You must reassign:
json_df = json_df.withColumn("new_col", lit(1))
```

### 5d. drop -- remove columns

You'll use this in the Silver layer to remove PII columns after hashing.

In [26]:
# Drop PII columns -- this is exactly what the Silver layer does
scrubbed = json_df.drop("national_id", "full_name", "phone_number")
scrubbed.show()
print(f"Columns remaining: {scrubbed.columns}")

+--------+---------+-------+-----------------+
|district|record_id|soil_ph|yield_estimate_kg|
+--------+---------+-------+-----------------+
|  Gasabo|     r001|    6.5|             2300|
|Kicukiro|     r002|    5.8|             1800|
|  Gasabo|     r003|    7.1|             3200|
| Musanze|     r004|    6.2|             2700|
+--------+---------+-------+-----------------+

Columns remaining: ['district', 'record_id', 'soil_ph', 'yield_estimate_kg']


### 5e. groupBy + agg -- aggregate data

You'll use this in the Gold layer to compute per-district statistics.

In [27]:
from pyspark.sql.functions import avg, sum as spark_sum, count

# Group by district, compute aggregates
gold = json_df.groupBy("district").agg(
    avg("soil_ph").alias("avg_soil_ph"),
    spark_sum("yield_estimate_kg").alias("total_yield_kg"),
    count("*").alias("record_count")
)
gold.show()

+--------+-----------+--------------+------------+
|district|avg_soil_ph|total_yield_kg|record_count|
+--------+-----------+--------------+------------+
|Kicukiro|        5.8|          1800|           1|
| Musanze|        6.2|          2700|           1|
|  Gasabo|        6.8|          5500|           2|
+--------+-----------+--------------+------------+



Notice `.alias()` -- without it, the column name would be `avg(soil_ph)` which is ugly and hard to reference later. Always alias aggregated columns.

### 5f. orderBy -- sort rows

In [28]:
# Sort by yield descending
json_df.orderBy(col("yield_estimate_kg").desc()).show()

+--------+--------------+-------------+-------------+---------+-------+-----------------+
|district|     full_name|  national_id| phone_number|record_id|soil_ph|yield_estimate_kg|
+--------+--------------+-------------+-------------+---------+-------+-----------------+
|  Gasabo| Marie Uwimana|1199580011111|+250782222222|     r003|    7.1|             3200|
| Musanze|Eric Niyonzima|1199580033333|+250784444444|     r004|    6.2|             2700|
|  Gasabo|Alice Mukamana|1199580012345|+250781234567|     r001|    6.5|             2300|
|Kicukiro| Jean Habimana|1199580067890|+250789876543|     r002|    5.8|             1800|
+--------+--------------+-------------+-------------+---------+-------+-----------------+



### YOUR TURN

Starting from `json_df`, build a chain that:
1. Drops `national_id`, `full_name`, `phone_number`
2. Adds a column `yield_tons` = `yield_estimate_kg / 1000`
3. Filters to only rows where `yield_tons > 2.0`
4. Sorts by `yield_tons` descending
5. Shows the result

Do it as a single chain (no intermediate variables).

In [ ]:
# YOUR TURN: single chain -- drop, withColumn, filter, orderBy, show



---
## 6. Joins

Joins combine two DataFrames on a common column. You'll use these less in the core pipeline but they're essential for enrichment steps (e.g., joining district codes to district names).

In [ ]:
# Create a reference table: district -> region mapping
regions = spark.createDataFrame([
    ("Gasabo", "Kigali"),
    ("Kicukiro", "Kigali"),
    ("Nyarugenge", "Kigali"),
    ("Musanze", "Northern"),
    ("Huye", "Southern"),
], ["district", "province"])

regions.show()

In [ ]:
# Inner join -- only rows where district matches in both DataFrames
joined = json_df.join(regions, on="district", how="inner")
joined.select("record_id", "district", "province", "soil_ph").show()

In [ ]:
# Left join -- keep all rows from the left DataFrame, even if no match
# Let's add a district that doesn't exist in our regions table
extra_row = spark.createDataFrame(
    [("r005", "1199580099999", "Paul Kagabo", "+250785555555", "Rubavu", 6.0, 2000)],
    json_df.columns
)
json_df_extended = json_df.union(extra_row)

left_joined = json_df_extended.join(regions, on="district", how="left")
left_joined.select("record_id", "district", "province").show()

Notice `Rubavu` has `null` for province -- it wasn't in the regions table. Left joins preserve all left-side rows.

---
## 7. User Defined Functions (UDFs)

PySpark has built-in functions for most things (`sha2`, `upper`, `avg`, etc.), but sometimes you need custom logic. UDFs let you apply any Python function to a column.

**Important tradeoff:** Built-in functions run inside Spark's JVM engine (fast). UDFs serialize data to Python and back (slower). Use built-ins whenever possible; UDFs for custom logic only.

In [ ]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import hashlib

# Define a Python function
def mask_phone(phone):
    if phone is None:
        return None
    return phone[:7] + "*" * (len(phone) - 7)

# Register it as a UDF
mask_phone_udf = udf(mask_phone, StringType())

# Apply it
masked = json_df.withColumn("masked_phone", mask_phone_udf(col("phone_number")))
masked.select("full_name", "phone_number", "masked_phone").show(truncate=False)

However, for SHA-256 hashing (which you'll need in the Silver layer), PySpark has a **built-in function** -- no UDF needed:

In [ ]:
from pyspark.sql.functions import sha2

# Built-in SHA-256 -- runs in the JVM, much faster than a UDF
hashed = json_df.withColumn("hashed_national_id", sha2(col("national_id"), 256))
hashed.select("national_id", "hashed_national_id").show(truncate=False)

The hash is a one-way function: you can't recover the original national ID from it. But the same input always produces the same hash, so you can still use it as a unique identifier for joins and deduplication -- you just can't reverse it to get the person's real ID.

This is the core of your Silver layer's PII scrubbing.

### YOUR TURN

Write a UDF that categorizes soil pH:
- pH < 5.5 -> `"acidic"`
- 5.5 <= pH <= 7.0 -> `"neutral"`
- pH > 7.0 -> `"alkaline"`

Apply it to `json_df` as a new column `ph_category` and show the result.

In [ ]:
# YOUR TURN: pH categorization UDF



**Bonus question:** Could you do this WITHOUT a UDF, using PySpark's built-in `when` function? Try it:

```python
from pyspark.sql.functions import when
# when(condition, value).when(condition, value).otherwise(value)
```

In [ ]:
# YOUR TURN (BONUS): pH categorization using when() -- no UDF



---
## 8. Writing Data

So far we've only read and transformed data. Let's write results to disk. This is an **action** -- it triggers execution of the full plan.

In [ ]:
# Write as Parquet (columnar binary format -- compressed, fast)
json_df.write.mode("overwrite").parquet("/home/jovyan/data/output/parquet_example")

# Read it back
spark.read.parquet("/home/jovyan/data/output/parquet_example").show()

In [ ]:
# Let's compare file sizes: JSON vs Parquet
import os

def dir_size(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total

json_size = os.path.getsize("/home/jovyan/data/raw/sample.json")
parquet_size = dir_size("/home/jovyan/data/output/parquet_example")

print(f"JSON:    {json_size:,} bytes")
print(f"Parquet: {parquet_size:,} bytes")
print(f"Ratio:   {json_size / parquet_size:.1f}x smaller as Parquet")

With only 4 rows the difference is small (Parquet has metadata overhead). With 1M rows, you'll see 5-10x compression. This is one of the benchmarks you'll measure in Phase 5.

**Write modes:**
- `"overwrite"` -- replace existing data
- `"append"` -- add to existing data (what Bronze ingestion uses)
- `"error"` (default) -- fail if data already exists
- `"ignore"` -- silently skip if data exists

---
## 9. Putting It All Together -- Mini Pipeline

Let's simulate the Bronze -> Silver transformation from your plan, using everything you've learned.

In [ ]:
# Step 1: Read raw data (Bronze)
bronze = spark.read.json("/home/jovyan/data/raw/sample.json")
print("=== BRONZE (raw data with PII) ===")
bronze.show(truncate=False)
print(f"Columns: {bronze.columns}")

In [ ]:
# Step 2: Transform to Silver (hash PII, drop raw PII, add metadata)
silver = bronze \
    .withColumn("hashed_national_id", sha2(col("national_id"), 256)) \
    .drop("national_id", "full_name", "phone_number") \
    .withColumn("processed_at", current_timestamp()) \
    .withColumn("compliance_flag", lit("RW_LAW_058_2021_COMPLIANT"))

print("=== SILVER (PII removed, hashed ID added) ===")
silver.show(truncate=False)
print(f"Columns: {silver.columns}")

In [ ]:
# Step 3: Verify -- are there any PII columns left?
pii_columns = {"national_id", "full_name", "phone_number"}
remaining_pii = pii_columns.intersection(set(silver.columns))

if remaining_pii:
    print(f"WARNING: PII columns still present: {remaining_pii}")
else:
    print("PASS: No PII columns in Silver table.")

This is exactly what `src/pipeline/02_scrub_silver.py` will do -- but reading from Delta format instead of JSON, and writing to Delta format instead of just displaying.

### YOUR TURN -- Gold Layer

Starting from the `silver` DataFrame above, build the Gold layer:
1. Group by `district`
2. Compute: `avg(soil_ph)` as `avg_soil_ph`, `sum(yield_estimate_kg)` as `total_yield_kg`, `count(*)` as `record_count`
3. Show the result

This is what an external AI developer would receive through Delta Sharing -- aggregated, anonymized, no PII.

In [ ]:
# YOUR TURN: Build the Gold aggregation



---
## 10. Checkpoint

Before moving to the Delta Lake notebook, make sure you can answer these in your own words.

Write your answers in the cells below (as markdown or comments) -- this is for your own understanding, not grading.

**Q1:** What is the difference between a transformation and an action? Give 3 examples of each.

*Your answer here*

**Q2:** Why does Spark use lazy evaluation instead of executing immediately? What advantage does it give?

*Your answer here*

**Q3:** What does `.explain()` show you, and why is it useful?

*Your answer here*

**Q4:** When should you use a UDF vs a built-in function like `sha2` or `when`? What's the performance difference?

*Your answer here*

**Q5:** Why are PySpark DataFrames immutable? What does that mean for how you write code?

*Your answer here*

---

**Next:** `02_delta_lake_intro.ipynb` -- adding transactional guarantees, time travel, and schema enforcement on top of what you just learned.